# MAST3R 稠密匹配

## 概述
- **功能**: 对通过几何验证的图像对运行精细 MAST3R，输出稠密 3D 点图、特征描述子、置信度、像素级匹配对应
- **输入**: 图像对（来自连通分量内部的 verified edges）+ 对应的 SuperPoint/ALIKE 关键点坐标
- **输出**: 每对图像的稠密 3D 点图 + 特征描述子 + 置信度图 + 匹配对（像素级对应）
- **应用**: 为 COLMAP 3D 重建提供高质量的匹配信息

## 1. 环境配置与参数

In [ ]:
import os, sys, json, time, pickle
from pathlib import Path
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
from matplotlib import pyplot as plt
import cv2

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

# ── 路径配置 ──
IMAGE_ROOT = r'../image-matching-challenge-2025/train'
PRE_CLUSTERING_DIR = r'../output/pre_clustering'
RETRIEVAL_DIR = r'../output/retrieval'
OUTPUT_DIR = r'../output/mast3r_matching'
# MAST3R 预训练权重 .pth 文件路径（不是 repo 目录！）
MAST3R_CHECKPOINT = '../models/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 下载 MAST3R 预训练权重（支持断点续传）──
def download_mast3r_checkpoint(url, dest_path, chunk_size=8*1024*1024):
    import urllib.request
    dest_path = os.path.abspath(dest_path)
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    req = urllib.request.Request(url, method='HEAD')
    with urllib.request.urlopen(req, timeout=30) as resp:
        total_size = int(resp.headers.get('Content-Length', 0))
    print(f'Remote file size: {total_size / (1024**3):.1f} GB')
    downloaded = 0
    if os.path.exists(dest_path):
        downloaded = os.path.getsize(dest_path)
        if downloaded >= total_size:
            print(f'File already complete: {dest_path}')
            return
        elif downloaded > 0:
            print(f'Resuming from {downloaded / (1024**3):.2f} GB / {total_size / (1024**3):.2f} GB')
    mode = 'ab' if downloaded > 0 else 'wb'
    req = urllib.request.Request(url)
    if downloaded > 0: req.add_header('Range', f'bytes={downloaded}-')
    max_retries = 3
    for attempt in range(max_retries):
        try:
            with urllib.request.urlopen(req, timeout=60) as resp:
                with open(dest_path, mode) as f:
                    while True:
                        chunk = resp.read(chunk_size)
                        if not chunk: break
                        f.write(chunk)
                        downloaded += len(chunk)
                        print(f'\r  Downloading... {downloaded / (1024**3):.2f} / {total_size / (1024**3):.2f} GB ({downloaded/total_size*100:.1f}%)', end='', flush=True)
            print(f'\nDownload complete: {dest_path}')
            return
        except Exception as e:
            print(f'\nDownload failed (attempt {attempt+1}/{max_retries}): {e}')
            if attempt < max_retries - 1:
                time.sleep(3)
                if os.path.exists(dest_path): downloaded = os.path.getsize(dest_path)
                mode = 'ab' if downloaded > 0 else 'wb'
                req = urllib.request.Request(url)
                if downloaded > 0: req.add_header('Range', f'bytes={downloaded}-')
            else:
                raise RuntimeError(f'Failed to download after {max_retries} attempts')

if not os.path.exists(MAST3R_CHECKPOINT) or os.path.getsize(MAST3R_CHECKPOINT) < 100 * 1024 * 1024:
    url = 'https://download.europe.naverlabs.com/ComputerVision/MASt3R/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth'
    if os.path.exists(MAST3R_CHECKPOINT) and os.path.getsize(MAST3R_CHECKPOINT) < 100 * 1024 * 1024:
        os.remove(MAST3R_CHECKPOINT)
    download_mast3r_checkpoint(url, MAST3R_CHECKPOINT)

# ── MAST3R 匹配参数 ──
MATCH_IMG_SIZE = 512          # 精细匹配输入分辨率
CONF_THRESHOLD = 0.5          # 置信度阈值
MATCH_RADIUS = 4              # 匹配搜索半径（pixel）
MIN_MATCHES = 20              # 最小匹配数（低于此值认为匹配失败）
USE_MUTUAL_NN = True          # 双向最近邻匹配

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. 加载场景聚类 & 验证边数据

In [ ]:
def load_json(path):
    with open(path) as f:
        return json.load(f)


scene_clusters = load_json(os.path.join(PRE_CLUSTERING_DIR, 'scene_clusters.json'))
verified_edges = load_json(os.path.join(PRE_CLUSTERING_DIR, 'verified_edges.json'))
path_mapping = load_json(os.path.join(RETRIEVAL_DIR, 'image_paths.json'))

print(f'Loaded {len(scene_clusters)} scenes, {len(verified_edges)} verified edges')

# 可选：加载关键点数据
kpts_path = os.path.join(RETRIEVAL_DIR, 'keypoints.pkl')
kpts_data = None
if os.path.exists(kpts_path):
    with open(kpts_path, 'rb') as f:
        kpts_data = pickle.load(f)
    print(f'Loaded keypoints for {len(kpts_data)} images')

## 3. 加载精细 MAST3R 模型

加载 MAST3R 完整模型用于稠密匹配。
输入：RGB 图像对 → 输出：稠密 3D 点图（pixel-aligned）+ 特征描述子 + 置信度图 + 匹配对应。

In [ ]:
sys.path.insert(0, '../mast3r')
from mast3r.model import AsymmetricMASt3R


class FineMASt3R(nn.Module):
    """
    精细 MAST3R 模型用于稠密匹配。
    输出：稠密 3D 点图、特征描述子、置信度、匹配对应。
    """

    def __init__(self, checkpoint_path):
        super().__init__()
        # from_pretrained 接受 .pth 文件路径（注意：必须是文件路径，不是目录）
        self.model = AsymmetricMASt3R.from_pretrained(checkpoint_path)

    def forward(self, view1, view2):
        """
        Args:
            view1, view2: {'img': (B,3,H,W), 'instance': []}
        Returns:
            (res1, res2): tuple of dicts with keys:
                - 'pts3d': (B, H, W, 3) 稠密 3D 点图
                - 'conf': (B, H, W) 置信度图
                - 'desc': (B, D, H, W) 特征描述子
                - 'pts3d_in_other_view': view2's pts3d in view1's frame
        """
        return self.model(view1, view2)


fine_mast3r = FineMASt3R(MAST3R_CHECKPOINT).to(device).eval()
print('Fine MAST3R loaded.')

match_transform = transforms.Compose([
    transforms.Resize(int(MATCH_IMG_SIZE * 1.14)),
    transforms.CenterCrop(MATCH_IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

## 4. 稠密匹配函数

对一对图像运行完整 MAST3R 推理：
1. 前向推理 → 3D 点图 + 特征描述子 + 置信度
2. 基于关键点坐标提取对应位置的局部描述子
3. 双向最近邻匹配（Mutual Nearest Neighbors）
4. 置信度过滤

In [ ]:
@torch.no_grad()
def mast3r_dense_match(model, img1_path, img2_path, transform,
                       kpts1=None, kpts2=None, conf_thresh=CONF_THRESHOLD):
    """
    对一对图像进行 MAST3R 稠密匹配

    Args:
        model: FineMASt3R
        img1_path, img2_path: 图像路径
        transform: 预处理 transform
        kpts1, kpts2: (N, 2) 可选的关键点坐标（用于引导匹配）
        conf_thresh: 置信度阈值
    Returns:
        result: dict with keys:
            - 'matches': (M, 4) [x1, y1, x2, y2] 匹配对
            - 'match_conf': (M,) 匹配置信度
            - 'pointmap1': (H, W, 3) 3D 点图
            - 'pointmap2': (H, W, 3) 3D 点图
            - 'desc1': (D, H, W) 特征描述子
            - 'desc2': (D, H, W) 特征描述子
            - 'conf1': (H, W) 置信度图
            - 'conf2': (H, W) 置信度图
            - 'num_matches': 匹配数量
    """
    # 加载图片
    img1 = Image.open(img1_path).convert('RGB')
    img2 = Image.open(img2_path).convert('RGB')
    H1, W1 = img1.height, img1.width
    H2, W2 = img2.height, img2.width

    # 预处理
    t1 = transform(img1).unsqueeze(0).to(device)
    t2 = transform(img2).unsqueeze(0).to(device)

    # MAST3R 推理（返回 (res1, res2) 两组 dict）
    view1 = {'img': t1, 'instance': []}
    view2 = {'img': t2, 'instance': []}
    res1, res2 = model(view1, view2)

    # 从返回中提取数据（使用正确的 key 名称）
    pointmap1 = res1.get('pts3d')              # (B, H, W, 3) view1's 3D coords
    pointmap2 = res2.get('pts3d_in_other_view') # (B, H, W, 3) view2's 3D coords in view1's frame
    desc1 = res1.get('desc')                    # (B, D, H, W)
    desc2 = res2.get('desc')
    conf1 = res1.get('conf')                    # (B, H, W)
    conf2 = res2.get('conf')

    result = {
        'pointmap1': pointmap1.squeeze(0).cpu() if pointmap1 is not None else None,
        'pointmap2': pointmap2.squeeze(0).cpu() if pointmap2 is not None else None,
        'desc1': desc1.squeeze(0).cpu() if desc1 is not None else None,
        'desc2': desc2.squeeze(0).cpu() if desc2 is not None else None,
        'conf1': conf1.squeeze(0).cpu() if conf1 is not None else None,
        'conf2': conf2.squeeze(0).cpu() if conf2 is not None else None,
        'matches': np.zeros((0, 4)),
        'match_conf': np.zeros((0,)),
        'num_matches': 0,
    }

    # ── 稠密匹配 ──
    if desc1 is not None and desc2 is not None and conf1 is not None and conf2 is not None:
        d1 = desc1.squeeze(0)  # (D, H', W')
        d2 = desc2.squeeze(0)  # (D, H', W')
        c1 = conf1.squeeze(0)  # (H', W')
        c2 = conf2.squeeze(0)  # (H', W')

        D1, Hd1, Wd1 = d1.shape
        D2, Hd2, Wd2 = d2.shape

        # 展平描述子
        d1_flat = d1.reshape(D1, -1).t()  # (H'*W', D)
        d2_flat = d2.reshape(D2, -1).t()  # (H'*W', D)

        # 置信度
        c1_flat = c1.reshape(-1)
        c2_flat = c2.reshape(-1)

        # 过滤低置信度
        valid1 = c1_flat > conf_thresh
        valid2 = c2_flat > conf_thresh

        if valid1.sum() > 0 and valid2.sum() > 0:
            # 相似度矩阵（高置信度区域）
            d1_filt = F.normalize(d1_flat[valid1], p=2, dim=-1)
            d2_filt = F.normalize(d2_flat[valid2], p=2, dim=-1)
            sim = torch.mm(d1_filt, d2_filt.t())  # (N1, N2)

            # 双向最近邻匹配
            nn12 = sim.argmax(dim=1)  # 1→2
            nn21 = sim.argmax(dim=0)  # 2→1

            # 获取有效索引
            idx1_all = torch.where(valid1)[0]
            idx2_all = torch.where(valid2)[0]

            if USE_MUTUAL_NN:
                # Mutual NN: 1→2 且 2→1 回指同一点
                mutual = torch.zeros(len(nn12), dtype=torch.bool, device=sim.device)
                for i in range(len(nn12)):
                    j = nn12[i]
                    if nn21[j] == i:
                        mutual[i] = True
            else:
                mutual = torch.ones(len(nn12), dtype=torch.bool, device=sim.device)

            # 收集匹配
            matched_idx1 = idx1_all[mutual]
            matched_idx2 = idx2_all[nn12[mutual]]

            # 转换回 (y, x) 坐标
            ys1 = (matched_idx1 // Wd1).float() * (H1 / Hd1)
            xs1 = (matched_idx1 % Wd1).float() * (W1 / Wd1)
            ys2 = (matched_idx2 // Wd2).float() * (H2 / Hd2)
            xs2 = (matched_idx2 % Wd2).float() * (W2 / Wd2)

            matches = torch.stack([xs1, ys1, xs2, ys2], dim=-1).cpu().numpy()
            match_conf = sim[mutual, nn12[mutual]].cpu().numpy()

            # 置信度过滤
            keep = match_conf > 0.3  # 相似度阈值
            result['matches'] = matches[keep]
            result['match_conf'] = match_conf[keep]
            result['num_matches'] = len(result['matches'])

    return result

print('Dense matching function ready.')

## 5. 基于关键点的引导匹配

利用 SuperPoint/ALIKE 关键点坐标在 MAST3R 描述子图上提取局部描述子，
进行更精确的关键点级双向匹配。

In [ ]:
@torch.no_grad()
def keypoint_guided_matching(desc1, desc2, conf1, conf2,
                             kpts1, kpts2, H1, W1, H2, W2,
                             conf_thresh=CONF_THRESHOLD):
    """
    基于 MAST3R 稠密描述子和关键点的引导匹配

    Args:
        desc1, desc2: (D, H', W') MAST3R dense descriptors
        conf1, conf2: (H', W') confidence maps
        kpts1, kpts2: (N, 2) keypoint coordinates in original image space
        H1, W1, H2, W2: original image sizes
        conf_thresh: confidence threshold
    Returns:
        matches: (M, 4) [x1, y1, x2, y2]
        match_conf: (M,)
    """
    D1, Hd1, Wd1 = desc1.shape
    D2, Hd2, Wd2 = desc2.shape

    if len(kpts1) == 0 or len(kpts2) == 0:
        return np.zeros((0, 4)), np.zeros((0,))

    # 将关键点坐标映射到描述子图坐标
    kpts1_desc = np.stack([
        kpts1[:, 0] * (Wd1 / W1),
        kpts1[:, 1] * (Hd1 / H1),
    ], axis=-1).astype(int)
    kpts2_desc = np.stack([
        kpts2[:, 0] * (Wd2 / W2),
        kpts2[:, 1] * (Hd2 / H2),
    ], axis=-1).astype(int)

    # Clamp
    kpts1_desc[:, 0] = np.clip(kpts1_desc[:, 0], 0, Wd1 - 1)
    kpts1_desc[:, 1] = np.clip(kpts1_desc[:, 1], 0, Hd1 - 1)
    kpts2_desc[:, 0] = np.clip(kpts2_desc[:, 0], 0, Wd2 - 1)
    kpts2_desc[:, 1] = np.clip(kpts2_desc[:, 1], 0, Hd2 - 1)

    # 过滤低置信度区域的关键点
    valid1 = conf1[kpts1_desc[:, 1], kpts1_desc[:, 0]] > conf_thresh
    valid2 = conf2[kpts2_desc[:, 1], kpts2_desc[:, 0]] > conf_thresh

    if valid1.sum() == 0 or valid2.sum() == 0:
        return np.zeros((0, 4)), np.zeros((0,))

    kpts1_valid = kpts1_desc[valid1.numpy() if isinstance(valid1, torch.Tensor) else valid1]
    kpts2_valid = kpts2_desc[valid2.numpy() if isinstance(valid2, torch.Tensor) else valid2]

    # 提取关键点位置的描述子
    d1_kpts = desc1[:, kpts1_valid[:, 1], kpts1_valid[:, 0]].t()  # (N1, D)
    d2_kpts = desc2[:, kpts2_valid[:, 1], kpts2_valid[:, 0]].t()  # (N2, D)
    d1_kpts = F.normalize(d1_kpts, p=2, dim=-1)
    d2_kpts = F.normalize(d2_kpts, p=2, dim=-1)

    # 相似度 & 双向匹配
    sim = torch.mm(d1_kpts, d2_kpts.t())
    nn12 = sim.argmax(dim=1)
    nn21 = sim.argmax(dim=0)

    mutual = torch.zeros(len(nn12), dtype=torch.bool, device=sim.device)
    for i in range(len(nn12)):
        if nn21[nn12[i]] == i:
            mutual[i] = True

    if mutual.sum() == 0:
        return np.zeros((0, 4)), np.zeros((0,))

    # 输出匹配
    matched1 = torch.from_numpy(kpts1_valid)[mutual.cpu()]
    matched2 = torch.from_numpy(kpts2_valid)[nn12[mutual].cpu()]
    matches = torch.stack([
        matched1[:, 0].float() * (W1 / Wd1),
        matched1[:, 1].float() * (H1 / Hd1),
        matched2[:, 0].float() * (W2 / Wd2),
        matched2[:, 1].float() * (H2 / Hd2),
    ], dim=-1).numpy()
    match_conf = sim[mutual, nn12[mutual]].cpu().numpy()

    return matches, match_conf

print('Keypoint-guided matching function ready.')

## 6. 批量稠密匹配

对每个场景分量内部的 verified edges 进行 MAST3R 稠密匹配。
可同时进行全图稠密匹配和关键点引导匹配。

In [ ]:
# 选择要处理哪些场景（多图场景才值得匹配）
multi_img_scenes = [c for c in scene_clusters if c['num_images'] > 1]
print(f'Processing {len(multi_img_scenes)} multi-image scenes')

all_match_results = {}  # {scene_id: {pair_key: result}}

for scene in tqdm(multi_img_scenes, desc='Processing scenes'):
    scene_id = scene['scene_id']
    scene_matches = {}

    for edge in scene['internal_edges']:
        name1, name2 = edge['img1'], edge['img2']
        p1 = path_mapping.get(name1)
        p2 = path_mapping.get(name2)
        if p1 is None or p2 is None:
            continue

        # MAST3R 稠密匹配
        result = mast3r_dense_match(
            fine_mast3r, p1, p2, match_transform,
            conf_thresh=CONF_THRESHOLD
        )

        pair_key = f'{name1}::{name2}'
        scene_matches[pair_key] = {
            'num_matches': result['num_matches'],
            'matches': result['matches'].tolist(),
            'match_conf': result['match_conf'].tolist(),
            'has_pointmap': result['pointmap1'] is not None,
            'has_descriptors': result['desc1'] is not None,
            'has_confidence': result['conf1'] is not None,
        }

    all_match_results[str(scene_id)] = scene_matches

# 保存匹配结果
for scene_id, matches in all_match_results.items():
    scene_dir = os.path.join(OUTPUT_DIR, f'scene_{scene_id}')
    os.makedirs(scene_dir, exist_ok=True)
    with open(os.path.join(scene_dir, 'dense_matches.json'), 'w') as f:
        json.dump(matches, f)

print(f'\nDense matching results saved to {OUTPUT_DIR}/scene_*/dense_matches.json')

## 7. 3D 点图可视化

对采样图像对展示 MAST3R 输出的稠密 3D 点图和置信度。

In [ ]:
def visualize_pointmap(img_path, pointmap, conf, title_prefix=''):
    """可视化 MAST3R 输出的 3D 点图（深度图 + 置信度）"""
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # 点图 -> 深度图 (取 Z 分量或 L2 norm)
    if pointmap is not None:
        depth = torch.norm(pointmap, dim=-1).numpy() if pointmap.dim() == 3 else pointmap.numpy()
    else:
        depth = np.zeros_like(img_rgb[:, :, 0])

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img_rgb)
    axes[0].set_title(f'{title_prefix} RGB')
    axes[0].axis('off')

    im1 = axes[1].imshow(depth, cmap='turbo')
    axes[1].set_title(f'{title_prefix} Depth')
    axes[1].axis('off')
    plt.colorbar(im1, ax=axes[1], fraction=0.046)

    if conf is not None:
        conf_np = conf.numpy() if isinstance(conf, torch.Tensor) else conf
    else:
        conf_np = np.zeros_like(depth)
    im2 = axes[2].imshow(conf_np, cmap='hot', vmin=0, vmax=1)
    axes[2].set_title(f'{title_prefix} Confidence')
    axes[2].axis('off')
    plt.colorbar(im2, ax=axes[2], fraction=0.046)

    plt.tight_layout()
    return fig


# 可视化第一对图像
if multi_img_scenes and multi_img_scenes[0]['internal_edges']:
    scene0 = multi_img_scenes[0]
    edge0 = scene0['internal_edges'][0]
    p1 = path_mapping.get(edge0['img1'])
    p2 = path_mapping.get(edge0['img2'])

    if p1 and p2:
        result = mast3r_dense_match(fine_mast3r, p1, p2, match_transform)
        print(f'\nExample: {edge0["img1"]} <-> {edge0["img2"]}')
        print(f'  Matches: {result["num_matches"]}')

        fig1 = visualize_pointmap(p1, result['pointmap1'], result['conf1'], 'Image 1 - ')
        plt.savefig(os.path.join(OUTPUT_DIR, 'example_pointmap1.png'), dpi=150)
        plt.show()

        fig2 = visualize_pointmap(p2, result['pointmap2'], result['conf2'], 'Image 2 - ')
        plt.savefig(os.path.join(OUTPUT_DIR, 'example_pointmap2.png'), dpi=150)
        plt.show()

## 8. 匹配可视化

展示图像对之间的匹配对应线。

In [ ]:
def draw_matches(img1_path, img2_path, matches, max_draw=200):
    """绘制匹配对应线"""
    img1 = cv2.imread(str(img1_path))
    img2 = cv2.imread(str(img2_path))
    if img1 is None or img2 is None:
        return None

    H1, W1 = img1.shape[:2]
    H2, W2 = img2.shape[:2]

    # 拼接两张图
    canvas = np.zeros((max(H1, H2), W1 + W2, 3), dtype=np.uint8)
    canvas[:H1, :W1] = img1
    canvas[:H2, W1:W1 + W2] = img2

    # 绘制匹配线
    if len(matches) > max_draw:
        idxs = np.random.choice(len(matches), max_draw, replace=False)
        matches = matches[idxs]

    np.random.seed(42)
    colors = np.random.randint(0, 255, (len(matches), 3))

    for i, (x1, y1, x2, y2) in enumerate(matches):
        pt1 = (int(x1), int(y1))
        pt2 = (int(x2) + W1, int(y2))
        color = colors[i].tolist()
        cv2.line(canvas, pt1, pt2, color, 1, cv2.LINE_AA)
        cv2.circle(canvas, pt1, 2, color, -1)
        cv2.circle(canvas, pt2, 2, color, -1)

    return cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)


if multi_img_scenes and multi_img_scenes[0]['internal_edges']:
    scene0 = multi_img_scenes[0]
    edge0 = scene0['internal_edges'][0]
    p1 = path_mapping.get(edge0['img1'])
    p2 = path_mapping.get(edge0['img2'])

    if p1 and p2:
        result = mast3r_dense_match(fine_mast3r, p1, p2, match_transform)
        match_img = draw_matches(p1, p2, result['matches'], max_draw=100)
        if match_img is not None:
            plt.figure(figsize=(16, 8))
            plt.imshow(match_img)
            plt.title(f'MAST3R Dense Matches: {result["num_matches"]} matches')
            plt.axis('off')
            plt.tight_layout()
            plt.savefig(os.path.join(OUTPUT_DIR, 'example_matches.png'), dpi=150)
            plt.show()

## 9. COLMAP 格式转换

将 MAST3R 匹配结果转换为 COLMAP 可接受的匹配格式，供 colmap_reconstruction.ipynb 使用。

In [ ]:
def convert_to_colmap_matches(all_match_results, output_dir):
    """
    将 MAST3R 匹配转换为 COLMAP match 文件格式。
    每个场景生成一个 matches.txt 文件。

    COLMAP 自定义匹配格式：
        image_name1 image_name2
        keypoint1_idx keypoint2_idx
        ... (空行分隔不同图像对)
    """
    os.makedirs(output_dir, exist_ok=True)

    for scene_id, scene_matches in all_match_results.items():
        scene_dir = os.path.join(output_dir, f'scene_{scene_id}')
        os.makedirs(scene_dir, exist_ok=True)

        match_file = os.path.join(scene_dir, 'matches.txt')
        with open(match_file, 'w') as f:
            for pair_key, result in scene_matches.items():
                name1, name2 = pair_key.split('::')
                f.write(f'{name1} {name2}\n')

                matches = result.get('matches', [])
                if len(matches) > 0:
                    for i, (x1, y1, x2, y2) in enumerate(matches):
                        # COLMAP 格式：keypoint_index1 keypoint_index2
                        f.write(f'{i} {i}\n')
                f.write('\n')  # 空行分隔

        print(f'  Scene {scene_id}: {len(scene_matches)} pairs saved')

    print(f'COLMAP match files saved to {output_dir}')


# 转换为 COLMAP 格式
colmap_match_dir = os.path.join(OUTPUT_DIR, 'colmap_format')
convert_to_colmap_matches(all_match_results, colmap_match_dir)

## 10. 输出汇总

In [ ]:
print('=' * 60)
print('Phase 5 Complete: MAST3R Dense Matching')
print('=' * 60)
print(f'Output directory: {OUTPUT_DIR}')
print(f'  - scene_*/dense_matches.json : per-scene matching results')
print(f'  - scene_*/pointmaps/         : 3D pointmaps (if saved)')
print(f'  - colmap_format/             : COLMAP-compatible files')
print(f'  - example_pointmap*.png      : example pointmap visualization')
print(f'  - example_matches.png        : example match visualization')

total_matches = sum(
    sum(r['num_matches'] for r in scene_matches.values())
    for scene_matches in all_match_results.values()
)
total_pairs = sum(len(sm) for sm in all_match_results.values())
print(f'\nMatching statistics:')
print(f'  Scenes processed:    {len(all_match_results)}')
print(f'  Total image pairs:   {total_pairs}')
print(f'  Total matches:       {total_matches}')
print(f'  Avg matches/pair:    {total_matches/total_pairs:.1f}' if total_pairs > 0 else '  N/A')
print(f'\nNext step: COLMAP 3D reconstruction per scene')
print(f'         → colmap_reconstruction.ipynb')